# Cheap-Talk Benchmark - Kaggle Runner

Runs the full Sabani/Georgousis-aligned sweep (5 runs x 16 rounds x {PD,SH} x {no_comm,cheap_talk}) for ONE model on Kaggle's free 30 h/week T4 GPU.

**Before running:**
1. Settings (right panel) -> Accelerator -> `GPU T4 x2` (or `GPU P100`)
2. Settings -> Internet -> `On`
3. Add Kaggle Secret named `HF_TOKEN` if you'll use a gated model (Llama, Gemma)
4. Edit the `MODEL` and `GITHUB_REPO` variables in cells 2 and 4 below

## Cell 1 - Install only what is missing on Kaggle

Kaggle preinstalls torch + transformers + accelerate + openai. Reinstalling them breaks the kernel (circular import in torch). Only `bitsandbytes` and `python-dotenv` are missing.

In [ ]:
!pip install -q bitsandbytes python-dotenv

import torch
assert torch.cuda.is_available(), "GPU not enabled! Settings -> Accelerator -> GPU T4 x2"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

import transformers
print(f"torch: {torch.__version__}  transformers: {transformers.__version__}")

## Cell 2 - Clone the benchmark code from GitHub

Note: the repo currently has files inside a `cheaptalk_bench/` subfolder. We `cd` into it after clone.

In [ ]:
GITHUB_REPO = "https://github.com/stsimpe/cheaptalk_bench.git"

import os
if not os.path.exists('/kaggle/working/repo'):
    !git clone $GITHUB_REPO /kaggle/working/repo
%cd /kaggle/working/repo/cheaptalk_bench
!ls

## Cell 3 - Load HF token (only needed for gated models: Llama, Gemma)

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HUGGINGFACE_API_KEY'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF token loaded from Kaggle Secrets')
except Exception as e:
    print(f'No HF_TOKEN secret found (fine for Qwen models): {e}')
    print('For gated models, add via Kaggle -> Add-ons -> Secrets -> HF_TOKEN')

## Cell 4 - Pick the model

In [ ]:
# Pick ONE model per session. The model loads ONCE in VRAM and stays.

# NOT gated -- start with Qwen for the smoke test:
MODEL = "Qwen/Qwen2.5-7B-Instruct"
# MODEL = "Qwen/Qwen2.5-3B-Instruct"
# MODEL = "Qwen/Qwen2.5-14B-Instruct"
# MODEL = "Qwen/Qwen3-4B"
# MODEL = "Qwen/Qwen3-8B"
# MODEL = "Qwen/Qwen3-14B"

# Gated by Google -- click Acknowledge License on each model's HF page first:
# MODEL = "google/gemma-2-2b-it"
# MODEL = "google/gemma-2-9b-it"
# MODEL = "google/gemma-3-4b-it"

# Gated by Meta -- request access on the HF page (~24h Meta approval):
# MODEL = "meta-llama/Llama-3.1-8B-Instruct"

OUT_DIR = f"results/{MODEL.split('/')[-1]}"
print(f"Will run sweep for {MODEL} -> {OUT_DIR}")

## Cell 5 - Smoke test (5-10 min)

2 runs x 8 rounds x 4 conditions = 192 calls. Verifies that model loads, gating works, and the JSON parser works on this model's output. Comment out once you trust the setup.

In [ ]:
!python run_full_sweep.py --provider local --model-id $MODEL --out-dir $OUT_DIR --quick --no-probe

## Cell 6 - Full Sabani-aligned sweep

5 runs x 16 rounds x {pd,sh} x {no_comm,cheap_talk} = 1,920 calls per model. Wall-time on T4:
- 3-4B model:  ~30 min
- 7-9B model:  ~1.5-2 h
- 14B model:   ~3 h

In [ ]:
!python run_full_sweep.py --provider local --model-id $MODEL --out-dir $OUT_DIR --no-probe

## Cell 7 - Pack results for download

In [ ]:
import shutil
model_short = MODEL.split('/')[-1]
zip_path = f"/kaggle/working/results_{model_short}"
shutil.make_archive(zip_path, 'zip', OUT_DIR)
print(f"Done. Zip created at {zip_path}.zip")
!ls -lh /kaggle/working/*.zip

## Cell 8 - Optional: on-Kaggle analysis

In [ ]:
!python analysis.py --results-dir $OUT_DIR